# Healthcare Claims Medallion Architecture Pipeline

This notebook implements a **batch processing** medallion architecture for HealthVerity claims sample data.

## Architecture Layers:

### 🥉 Bronze (Source)
- **Catalog/Schema**: `healthverity_claims_sample_patient_dataset.hv_claims_sample`
- **Tables**: medical_claim, diagnosis, procedure, provider, enrollment
- **Description**: Raw source data as-is

### 🥈 Silver Layer
- **Catalog/Schema**: `eli_lilly_demo.silver_claims`
- **Transformations**: Data cleansing, deduplication, standardization, data quality checks
- **Metadata**: Added processed_timestamp and data_quality_flag columns

### 🥇 Gold Layer
- **Catalog/Schema**: `eli_lilly_demo.gold_claims`
- **Analytics Tables**: 
  - `patient_summary` - Patient demographics and enrollment analytics
  - `claims_analytics` - Claims aggregated by date, location, procedure
  - `cost_analytics` - Financial analysis of charges vs allowed amounts

## 1. Setup and Configuration

In [0]:
# Import required libraries
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

# Configuration
BRONZE_CATALOG = "healthverity_claims_sample_patient_dataset"
BRONZE_SCHEMA = "hv_claims_sample"
TARGET_CATALOG = "eli_lilly_demo"
SILVER_SCHEMA = "silver_claims"
GOLD_SCHEMA = "gold_claims"

print(f"Source: {BRONZE_CATALOG}.{BRONZE_SCHEMA}")
print(f"Silver: {TARGET_CATALOG}.{SILVER_SCHEMA}")
print(f"Gold: {TARGET_CATALOG}.{GOLD_SCHEMA}")

Source: healthverity_claims_sample_patient_dataset.hv_claims_sample
Silver: eli_lilly_demo.silver_claims
Gold: eli_lilly_demo.gold_claims


In [0]:
%sql
-- Create Silver and Gold schemas in eli_lilly_demo catalog

CREATE SCHEMA IF NOT EXISTS eli_lilly_demo.silver_claims
COMMENT 'Silver layer: Cleaned and standardized healthcare claims data';

CREATE SCHEMA IF NOT EXISTS eli_lilly_demo.gold_claims
COMMENT 'Gold layer: Aggregated analytics for healthcare claims';

## 2. Silver Layer - Data Cleansing and Standardization

Transformations applied:
- Remove duplicates
- Trim whitespace from string columns
- Handle null values
- Validate date ranges
- Add metadata columns (processed_timestamp, data_quality_flag)

In [0]:
# Bronze to Silver: medical_claim table
# Transformations: deduplication, null handling, date validation

medical_claim_bronze = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.medical_claim")

# Apply transformations
medical_claim_silver = (
    medical_claim_bronze
    # Trim string columns
    .withColumn("claim_id", F.trim(F.col("claim_id")))
    .withColumn("patient_id", F.trim(F.col("patient_id")))
    .withColumn("location_of_care", F.trim(F.col("location_of_care")))
    .withColumn("pay_type", F.trim(F.col("pay_type")))
    # Data quality flag: check for nulls in critical fields and date validity
    .withColumn(
        "data_quality_flag",
        F.when(
            (F.col("claim_id").isNull()) | 
            (F.col("patient_id").isNull()) | 
            (F.col("date_service").isNull()) |
            (F.col("date_service_end") < F.col("date_service")),
            "FAIL"
        ).otherwise("PASS")
    )
    # Add processing timestamp
    .withColumn("processed_timestamp", F.current_timestamp())
    # Remove duplicates based on claim_id and patient_id
    .dropDuplicates(["claim_id", "patient_id"])
)

# Write to Silver layer
medical_claim_silver.write.mode("overwrite").saveAsTable(
    f"{TARGET_CATALOG}.{SILVER_SCHEMA}.medical_claim"
)

print(f"✓ Silver table created: {TARGET_CATALOG}.{SILVER_SCHEMA}.medical_claim")
print(f"  Records processed: {medical_claim_silver.count():,}")

✓ Silver table created: eli_lilly_demo.silver_claims.medical_claim
  Records processed: 152,156


In [0]:
# Bronze to Silver: diagnosis table
# Transformations: deduplication, standardization, null handling

diagnosis_bronze = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.diagnosis")

diagnosis_silver = (
    diagnosis_bronze
    # Trim string columns
    .withColumn("claim_id", F.trim(F.col("claim_id")))
    .withColumn("patient_id", F.trim(F.col("patient_id")))
    .withColumn("diagnosis_code", F.trim(F.upper(F.col("diagnosis_code"))))  # Standardize to uppercase
    .withColumn("diagnosis_qual", F.trim(F.col("diagnosis_qual")))
    .withColumn("admit_diagnosis_ind", F.trim(F.upper(F.col("admit_diagnosis_ind"))))
    # Data quality checks
    .withColumn(
        "data_quality_flag",
        F.when(
            (F.col("claim_id").isNull()) | 
            (F.col("patient_id").isNull()) | 
            (F.col("diagnosis_code").isNull()) |
            (F.col("date_service_end") < F.col("date_service")),
            "FAIL"
        ).otherwise("PASS")
    )
    .withColumn("processed_timestamp", F.current_timestamp())
    # Remove duplicates
    .dropDuplicates(["claim_id", "patient_id", "diagnosis_code"])
)

diagnosis_silver.write.mode("overwrite").saveAsTable(
    f"{TARGET_CATALOG}.{SILVER_SCHEMA}.diagnosis"
)

print(f"✓ Silver table created: {TARGET_CATALOG}.{SILVER_SCHEMA}.diagnosis")
print(f"  Records processed: {diagnosis_silver.count():,}")

✓ Silver table created: eli_lilly_demo.silver_claims.diagnosis
  Records processed: 368,265


In [0]:
# Bronze to Silver: procedure table
# Transformations: deduplication, numeric validation, standardization

procedure_bronze = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.procedure")

procedure_silver = (
    procedure_bronze
    # Trim string columns
    .withColumn("claim_id", F.trim(F.col("claim_id")))
    .withColumn("patient_id", F.trim(F.col("patient_id")))
    .withColumn("service_line_number", F.trim(F.col("service_line_number")))
    .withColumn("procedure_code", F.trim(F.upper(F.col("procedure_code"))))
    .withColumn("procedure_qual", F.trim(F.col("procedure_qual")))
    .withColumn("revenue_code", F.trim(F.col("revenue_code")))
    # Standardize modifiers
    .withColumn("procedure_modifier1", F.trim(F.upper(F.col("procedure_modifier1"))))
    .withColumn("procedure_modifier2", F.trim(F.upper(F.col("procedure_modifier2"))))
    .withColumn("procedure_modifier3", F.trim(F.upper(F.col("procedure_modifier3"))))
    .withColumn("procedure_modifier4", F.trim(F.upper(F.col("procedure_modifier4"))))
    # Data quality checks: validate numeric fields and critical columns
    .withColumn(
        "data_quality_flag",
        F.when(
            (F.col("claim_id").isNull()) | 
            (F.col("patient_id").isNull()) | 
            (F.col("procedure_code").isNull()) |
            (F.col("date_service_end") < F.col("date_service")) |
            (F.col("line_charge") < 0) |
            (F.col("line_allowed") < 0),
            "FAIL"
        ).otherwise("PASS")
    )
    .withColumn("processed_timestamp", F.current_timestamp())
    # Remove duplicates
    .dropDuplicates(["claim_id", "patient_id", "service_line_number"])
)

procedure_silver.write.mode("overwrite").saveAsTable(
    f"{TARGET_CATALOG}.{SILVER_SCHEMA}.procedure"
)

print(f"✓ Silver table created: {TARGET_CATALOG}.{SILVER_SCHEMA}.procedure")
print(f"  Records processed: {procedure_silver.count():,}")

✓ Silver table created: eli_lilly_demo.silver_claims.procedure
  Records processed: 359,825


In [0]:
# Bronze to Silver: provider table
# Transformations: deduplication, NPI standardization

provider_bronze = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.provider")

provider_silver = (
    provider_bronze
    # Trim string columns
    .withColumn("claim_id", F.trim(F.col("claim_id")))
    .withColumn("patient_id", F.trim(F.col("patient_id")))
    .withColumn("npi", F.trim(F.col("npi")))
    .withColumn("npi_role", F.trim(F.upper(F.col("npi_role"))))
    .withColumn("taxonomy_code", F.trim(F.col("taxonomy_code")))
    # Data quality checks: NPI should be 10 digits
    .withColumn(
        "data_quality_flag",
        F.when(
            (F.col("claim_id").isNull()) | 
            (F.col("patient_id").isNull()) | 
            (F.col("npi").isNull()) |
            (F.length(F.col("npi")) != 10),
            "FAIL"
        ).otherwise("PASS")
    )
    .withColumn("processed_timestamp", F.current_timestamp())
    # Remove duplicates
    .dropDuplicates(["claim_id", "patient_id", "npi", "npi_role"])
)

provider_silver.write.mode("overwrite").saveAsTable(
    f"{TARGET_CATALOG}.{SILVER_SCHEMA}.provider"
)

print(f"✓ Silver table created: {TARGET_CATALOG}.{SILVER_SCHEMA}.provider")
print(f"  Records processed: {provider_silver.count():,}")

✓ Silver table created: eli_lilly_demo.silver_claims.provider
  Records processed: 465,516


In [0]:
# Bronze to Silver: enrollment table
# Transformations: deduplication, demographic standardization, date validation

enrollment_bronze = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.enrollment")

enrollment_silver = (
    enrollment_bronze
    # Trim string columns
    .withColumn("patient_id", F.trim(F.col("patient_id")))
    .withColumn("patient_gender", F.trim(F.upper(F.col("patient_gender"))))
    .withColumn("patient_year_of_birth", F.trim(F.col("patient_year_of_birth")))
    .withColumn("patient_zip3", F.trim(F.col("patient_zip3")))
    .withColumn("patient_state", F.trim(F.upper(F.col("patient_state"))))
    .withColumn("benefit_type", F.trim(F.upper(F.col("benefit_type"))))
    .withColumn("pay_type", F.trim(F.upper(F.col("pay_type"))))
    # Calculate age (approximate based on current year)
    .withColumn(
        "patient_age",
        F.when(
            F.col("patient_year_of_birth").isNotNull(),
            F.year(F.current_date()) - F.col("patient_year_of_birth").cast("int")
        )
    )
    # Data quality checks
    .withColumn(
        "data_quality_flag",
        F.when(
            (F.col("patient_id").isNull()) | 
            (F.col("date_start").isNull()) |
            (F.col("date_end") < F.col("date_start")) |
            (F.col("patient_age") < 0) |
            (F.col("patient_age") > 120),
            "FAIL"
        ).otherwise("PASS")
    )
    .withColumn("processed_timestamp", F.current_timestamp())
    # Remove duplicates
    .dropDuplicates(["patient_id", "date_start"])
)

enrollment_silver.write.mode("overwrite").saveAsTable(
    f"{TARGET_CATALOG}.{SILVER_SCHEMA}.enrollment"
)

print(f"✓ Silver table created: {TARGET_CATALOG}.{SILVER_SCHEMA}.enrollment")
print(f"  Records processed: {enrollment_silver.count():,}")

✓ Silver table created: eli_lilly_demo.silver_claims.enrollment
  Records processed: 1,956


## 3. Gold Layer - Aggregated Analytics

Business-ready analytics tables:
1. **patient_summary**: Patient demographics, enrollment periods, claims activity
2. **claims_analytics**: Claims trends by date, location, procedure type
3. **cost_analytics**: Financial metrics - charges vs allowed amounts by procedure/provider

In [0]:
# Gold Layer: Patient Summary Analytics
# Combines enrollment demographics with claims activity

# Read silver tables
enrollment_df = spark.table(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.enrollment")
medical_claim_df = spark.table(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.medical_claim")

# Aggregate claims per patient
claims_per_patient = (
    medical_claim_df
    .where(F.col("data_quality_flag") == "PASS")
    .groupBy("patient_id")
    .agg(
        F.count("*").alias("total_claims"),
        F.min("date_service").alias("first_claim_date"),
        F.max("date_service").alias("last_claim_date"),
        F.countDistinct("location_of_care").alias("distinct_locations"),
        F.countDistinct("pay_type").alias("distinct_pay_types")
    )
)

# Create patient summary by joining enrollment with claims
patient_summary_gold = (
    enrollment_df
    .where(F.col("data_quality_flag") == "PASS")
    .groupBy("patient_id")
    .agg(
        F.first("patient_gender").alias("gender"),
        F.first("patient_age").alias("age"),
        F.first("patient_zip3").alias("zip3"),
        F.first("patient_state").alias("state"),
        F.min("date_start").alias("enrollment_start"),
        F.max("date_end").alias("enrollment_end"),
        F.datediff(F.max("date_end"), F.min("date_start")).alias("enrollment_days"),
        F.countDistinct("benefit_type").alias("benefit_types_count"),
        F.collect_set("benefit_type").alias("benefit_types")
    )
    .join(claims_per_patient, "patient_id", "left")
    .withColumn("total_claims", F.coalesce(F.col("total_claims"), F.lit(0)))
    .withColumn("processed_timestamp", F.current_timestamp())
)

patient_summary_gold.write.mode("overwrite").saveAsTable(
    f"{TARGET_CATALOG}.{GOLD_SCHEMA}.patient_summary"
)

print(f"✓ Gold table created: {TARGET_CATALOG}.{GOLD_SCHEMA}.patient_summary")
print(f"  Total patients: {patient_summary_gold.count():,}")

✓ Gold table created: eli_lilly_demo.gold_claims.patient_summary
  Total patients: 1,485


In [0]:
# Gold Layer: Claims Analytics
# Aggregated claims by date, location, and procedure type

medical_claim_df = spark.table(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.medical_claim")
procedure_df = spark.table(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.procedure")
diagnosis_df = spark.table(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.diagnosis")

# Join claims with procedures and diagnosis
claims_enriched = (
    medical_claim_df
    .where(F.col("data_quality_flag") == "PASS")
    .join(
        procedure_df.where(F.col("data_quality_flag") == "PASS").drop("date_service", "date_service_end", "data_quality_flag", "processed_timestamp"),
        ["claim_id", "patient_id"],
        "left"
    )
    .join(
        diagnosis_df.where(F.col("data_quality_flag") == "PASS").drop("date_service", "date_service_end", "data_quality_flag", "processed_timestamp"),
        ["claim_id", "patient_id"],
        "left"
    )
)

# Aggregate by date and location
claims_analytics_gold = (
    claims_enriched
    .groupBy(
        F.year("date_service").alias("service_year"),
        F.month("date_service").alias("service_month"),
        "location_of_care",
        "pay_type"
    )
    .agg(
        F.count("claim_id").alias("total_claims"),
        F.countDistinct("patient_id").alias("unique_patients"),
        F.countDistinct("procedure_code").alias("unique_procedures"),
        F.countDistinct("diagnosis_code").alias("unique_diagnoses"),
        F.sum("line_charge").alias("total_charges"),
        F.sum("line_allowed").alias("total_allowed"),
        F.avg("line_charge").alias("avg_line_charge"),
        F.avg("line_allowed").alias("avg_line_allowed")
    )
    .withColumn(
        "charge_to_allowed_ratio",
        F.when(
            F.col("total_allowed") > 0,
            F.col("total_charges") / F.col("total_allowed")
        )
    )
    .withColumn("processed_timestamp", F.current_timestamp())
    .orderBy("service_year", "service_month", "location_of_care")
)

claims_analytics_gold.write.mode("overwrite").saveAsTable(
    f"{TARGET_CATALOG}.{GOLD_SCHEMA}.claims_analytics"
)

print(f"✓ Gold table created: {TARGET_CATALOG}.{GOLD_SCHEMA}.claims_analytics")
print(f"  Analytics periods: {claims_analytics_gold.count():,}")

✓ Gold table created: eli_lilly_demo.gold_claims.claims_analytics
  Analytics periods: 3,322


In [0]:
# Gold Layer: Cost Analytics
# Financial analysis by procedure and provider

procedure_df = spark.table(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.procedure")
provider_df = spark.table(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.provider")

# Join procedures with providers for cost analysis
cost_analytics_gold = (
    procedure_df
    .where(F.col("data_quality_flag") == "PASS")
    .join(
        provider_df.where(F.col("data_quality_flag") == "PASS"),
        ["claim_id", "patient_id"],
        "left"
    )
    .groupBy(
        "procedure_code",
        "procedure_qual",
        "npi",
        "taxonomy_code"
    )
    .agg(
        F.count("*").alias("procedure_count"),
        F.countDistinct("patient_id").alias("unique_patients"),
        F.sum("line_charge").alias("total_charges"),
        F.sum("line_allowed").alias("total_allowed"),
        F.avg("line_charge").alias("avg_charge_per_procedure"),
        F.avg("line_allowed").alias("avg_allowed_per_procedure"),
        F.min("line_charge").alias("min_charge"),
        F.max("line_charge").alias("max_charge"),
        F.stddev("line_charge").alias("stddev_charge")
    )
    .withColumn(
        "charge_to_allowed_ratio",
        F.when(
            F.col("total_allowed") > 0,
            F.col("total_charges") / F.col("total_allowed")
        )
    )
    .withColumn(
        "discount_amount",
        F.col("total_charges") - F.col("total_allowed")
    )
    .withColumn(
        "discount_percentage",
        F.when(
            F.col("total_charges") > 0,
            ((F.col("total_charges") - F.col("total_allowed")) / F.col("total_charges")) * 100
        )
    )
    .withColumn("processed_timestamp", F.current_timestamp())
    .orderBy(F.desc("procedure_count"))
)

cost_analytics_gold.write.mode("overwrite").saveAsTable(
    f"{TARGET_CATALOG}.{GOLD_SCHEMA}.cost_analytics"
)

print(f"✓ Gold table created: {TARGET_CATALOG}.{GOLD_SCHEMA}.cost_analytics")
print(f"  Procedure/Provider combinations: {cost_analytics_gold.count():,}")

✓ Gold table created: eli_lilly_demo.gold_claims.cost_analytics
  Procedure/Provider combinations: 190,995


## 4. Pipeline Summary & Validation

Validate the medallion architecture and check data quality metrics.

In [0]:
# Generate data quality summary across all Silver tables

silver_tables = ["medical_claim", "diagnosis", "procedure", "provider", "enrollment"]

print("\n" + "="*80)
print("SILVER LAYER - DATA QUALITY SUMMARY")
print("="*80 + "\n")

for table in silver_tables:
    df = spark.table(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.{table}")
    total_records = df.count()
    passed_records = df.where(F.col("data_quality_flag") == "PASS").count()
    failed_records = df.where(F.col("data_quality_flag") == "FAIL").count()
    pass_rate = (passed_records / total_records * 100) if total_records > 0 else 0
    
    print(f"Table: {table}")
    print(f"  Total Records: {total_records:,}")
    print(f"  Passed: {passed_records:,} ({pass_rate:.2f}%)")
    print(f"  Failed: {failed_records:,}")
    print()

print("="*80)


SILVER LAYER - DATA QUALITY SUMMARY

Table: medical_claim
  Total Records: 152,156
  Passed: 152,156 (100.00%)
  Failed: 0

Table: diagnosis
  Total Records: 368,265
  Passed: 368,265 (100.00%)
  Failed: 0

Table: procedure
  Total Records: 359,825
  Passed: 348,195 (96.77%)
  Failed: 11,630

Table: provider
  Total Records: 465,516
  Passed: 465,516 (100.00%)
  Failed: 0

Table: enrollment
  Total Records: 1,956
  Passed: 1,956 (100.00%)
  Failed: 0



In [0]:
# Summary of record counts across all layers

print("\n" + "="*80)
print("MEDALLION ARCHITECTURE - RECORD COUNTS")
print("="*80 + "\n")

# Bronze layer
print("🥉 BRONZE LAYER (Source)")
print(f"  Catalog.Schema: {BRONZE_CATALOG}.{BRONZE_SCHEMA}")
for table in ["medical_claim", "diagnosis", "procedure", "provider", "enrollment"]:
    count = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.{table}").count()
    print(f"  {table}: {count:,}")

print("\n🥈 SILVER LAYER (Cleansed)")
print(f"  Catalog.Schema: {TARGET_CATALOG}.{SILVER_SCHEMA}")
for table in ["medical_claim", "diagnosis", "procedure", "provider", "enrollment"]:
    count = spark.table(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.{table}").count()
    print(f"  {table}: {count:,}")

print("\n🥇 GOLD LAYER (Analytics)")
print(f"  Catalog.Schema: {TARGET_CATALOG}.{GOLD_SCHEMA}")
for table in ["patient_summary", "claims_analytics", "cost_analytics"]:
    count = spark.table(f"{TARGET_CATALOG}.{GOLD_SCHEMA}.{table}").count()
    print(f"  {table}: {count:,}")

print("\n" + "="*80)
print("✓ Pipeline completed successfully!")
print("="*80)


MEDALLION ARCHITECTURE - RECORD COUNTS

🥉 BRONZE LAYER (Source)
  Catalog.Schema: healthverity_claims_sample_patient_dataset.hv_claims_sample
  medical_claim: 301,748
  diagnosis: 368,265
  procedure: 406,042
  provider: 465,516
  enrollment: 3,497

🥈 SILVER LAYER (Cleansed)
  Catalog.Schema: eli_lilly_demo.silver_claims
  medical_claim: 152,156
  diagnosis: 368,265
  procedure: 359,825
  provider: 465,516
  enrollment: 1,956

🥇 GOLD LAYER (Analytics)
  Catalog.Schema: eli_lilly_demo.gold_claims
  patient_summary: 1,485
  claims_analytics: 3,322
  cost_analytics: 190,995

✓ Pipeline completed successfully!


In [0]:
%sql
-- Sample queries to explore the Gold layer analytics

-- Top 10 patients by claim count
SELECT 
  patient_id,
  gender,
  age,
  state,
  total_claims,
  enrollment_days,
  first_claim_date,
  last_claim_date
FROM eli_lilly_demo.gold_claims.patient_summary
WHERE total_claims > 0
ORDER BY total_claims DESC
LIMIT 10;

patient_id,gender,age,state,total_claims,enrollment_days,first_claim_date,last_claim_date
2e558ebb643f2e38831757de988f6d8c,F,92,WI,320,2615,2017-06-15,2023-12-14
35b540c47e12c2ee00961cc17430aaf1,M,22,OH,318,2615,2017-01-09,2023-12-23
657a3f15e3219c5be2a189e268c491fb,F,83,PA,318,2615,2017-01-06,2023-12-19
7d19a151bd37b1ef75b540686b78a39b,F,83,CA,317,2496,2017-04-20,2023-12-18
dc3e379a0615f8f5abcffe3ebee6f4a5,F,80,TX,317,1884,2018-02-07,2023-02-15
0a3e83e5fc70ce368803d9f3c16a42ce,M,40,RI,311,2615,2017-01-31,2023-11-10
b56d41a750796026e00a3083a5426317,M,75,MI,310,2555,2017-01-18,2023-11-21
56f10702e135da8196f47087e88987d4,F,86,PA,308,2615,2017-01-03,2023-12-29
e140a6d7c81ccbdbb86462aecd9f3fcc,M,92,PA,308,2615,2017-02-07,2023-12-28
53341d1545f5f198e6e6754538af0b62,F,77,TX,307,2190,2018-01-05,2023-12-22
